# Ôn tập Buổi 07 - String và Time Series

        **Thời lượng gợi ý:** 60 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Làm sạch chuỗi vectorized bằng `.str` và regex.
- Chuyển dữ liệu sang datetime an toàn.
- Phân biệt resample, shift và rolling.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi7_python_datascience.pdf`.


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import numpy as np
import pandas as pd
DATA_DIR = ROOT / 'datasets' / 'buoi6'
print(f'Repo: {ROOT}')


## 0. Chẩn đoán nhanh - chưa chạy code

**1. Vì sao dùng `.str` thay loop?**

<details><summary>Kiểm tra đáp án</summary>

Code ngắn, vectorized và xử lý missing nhất quán hơn.

</details>

**2. `errors='coerce'` trong `to_datetime` làm gì?**

<details><summary>Kiểm tra đáp án</summary>

Giá trị không chuyển được thành `NaT`, giúp phát hiện và xử lý tường minh.

</details>

**3. `rolling(7).mean()` và `resample('W').mean()` khác gì?**

<details><summary>Kiểm tra đáp án</summary>

Rolling là cửa sổ trượt trên quan sát; resample đổi tần suất theo các khoảng thời gian.

</details>


## 1. Chuẩn hóa chuỗi và regex


In [ ]:
contacts = pd.DataFrame({
    "name": ["  Nguyễn An ", "TRẦN BÌNH", None, "Lê Chi"],
    "email": ["AN@EXAMPLE.COM", "binh.example.com", None, "chi@school.edu.vn"],
})
contacts["name_clean"] = contacts["name"].str.strip().str.title()
contacts["email_clean"] = contacts["email"].str.strip().str.lower()
contacts["email_valid"] = contacts["email_clean"].str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", na=False)
contacts["domain"] = contacts["email_clean"].str.extract(r"@(.+)$", expand=False)
display(contacts)
assert contacts["email_valid"].tolist() == [True, False, False, True]


## 2. Multi-label bằng `str.get_dummies()`


In [ ]:
movies = pd.DataFrame({"title": ["A", "B", "C"], "genres": ["Drama|Romance", "Action|Drama", "Comedy"]})
genre_flags = movies["genres"].str.get_dummies(sep="|")
display(pd.concat([movies, genre_flags], axis=1))
assert genre_flags.loc[0, "Drama"] == 1


## 3. Tạo datetime và audit ngày lỗi


In [ ]:
births = pd.read_csv(DATA_DIR / "births.csv")
births["date"] = pd.to_datetime(births[["year", "month", "day"]], errors="coerce")
invalid_dates = births["date"].isna().sum()
print("invalid dates:", invalid_dates)
births_valid = births.dropna(subset=["date"]).set_index("date").sort_index()
assert isinstance(births_valid.index, pd.DatetimeIndex)


## 4. Resample: đổi tần suất thời gian


In [ ]:
daily_births = births_valid.groupby(level=0)["births"].sum()
yearly_births = daily_births.resample("YS").sum()
display(yearly_births.head())
assert yearly_births.index.is_monotonic_increasing


## 5. `shift()` và `rolling()`

- `shift(1)`: căn giá trị kỳ trước để tính thay đổi.
- `rolling(30)`: tóm tắt cửa sổ 30 quan sát liên tiếp.


In [ ]:
trend = pd.DataFrame({"births": daily_births})
trend["change_from_previous_day"] = trend["births"] - trend["births"].shift(1)
trend["rolling_30d_mean"] = trend["births"].rolling(30, min_periods=30).mean()
display(trend.head(35).tail())
assert trend["rolling_30d_mean"].first_valid_index() == trend.index[29]


## Bài tự luyện

        Từ `births_valid`, tính tổng births theo tháng, thêm phần trăm thay đổi so với tháng trước và rolling mean 12 tháng.

        <details><summary>Gợi ý / đáp án tham khảo</summary>

        ```python
        monthly = births_valid["births"].resample("MS").sum().to_frame()
monthly["pct_change"] = monthly["births"].pct_change()
monthly["rolling_12m"] = monthly["births"].rolling(12).mean()
        ```

        </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi dùng `.str` mà vẫn xử lý được missing.
- [ ] Tôi audit `NaT` sau `to_datetime(errors='coerce')`.
- [ ] Tôi phân biệt datetime index, resample, shift và rolling.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
